# 03 Degree Profile Construction

Purpose: combine degree-module mappings with cleaned module skill descriptions to create reusable degree-level profiles.

Inputs:
- `data/interim/modules_clean.parquet`
- `data/degree_to_module_map.csv`
- `data/nus_modules_skills_output.csv`

Outputs:
- `data/processed/degree_modules.parquet`
- `data/processed/degree_profiles.parquet`


In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.io import paths, require_files

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

from analysis.profiles import build_degree_profiles, prepare_degree_modules

require_files([paths.modules_clean_parquet, paths.degree_map_csv, paths.module_skills_csv])
print('Expected upstream inputs: cleaned modules plus degree mapping tables')


Expected upstream inputs: cleaned modules plus degree mapping tables


In [2]:
modules_clean = pd.read_parquet(paths.modules_clean_parquet)
degree_map = pd.read_csv(paths.degree_map_csv)
module_skills = pd.read_csv(paths.module_skills_csv)

degree_modules = prepare_degree_modules(degree_map, module_skills, modules_clean=modules_clean)
degree_profiles = build_degree_profiles(degree_modules, max_words=8000)

degree_modules.to_parquet(paths.degree_modules_parquet, index=False)
degree_profiles.to_parquet(paths.degree_profiles_parquet, index=False)

print(f'Saved degree modules to: {paths.degree_modules_parquet}')
print(f'Saved degree profiles to: {paths.degree_profiles_parquet}')
print(f'Degrees profiled: {len(degree_profiles):,}')


Saved degree modules to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/processed/degree_modules.parquet
Saved degree profiles to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/processed/degree_profiles.parquet
Degrees profiled: 5


In [3]:
sample_degree_id = 'cs' if (degree_profiles['degree_id'] == 'cs').any() else degree_profiles.iloc[0]['degree_id']
sample_profile = degree_profiles[degree_profiles['degree_id'] == sample_degree_id].iloc[0]
sample_modules = degree_modules[degree_modules['degree_id'] == sample_degree_id].copy()

print(f'Degree: {sample_profile["degree_name"]} ({sample_profile["degree_id"]})')
print(f'Modules used: {sample_profile["n_modules"]}')
print(f'Word count: {sample_profile["word_count"]}')
display(sample_modules[['requirement_group', 'moduleCode', 'title', 'top_skills']].head(15))
print(sample_profile['profile_text'][:2000])


Degree: Computer Science (cs)
Modules used: 23
Word count: 3211


,requirement_group,moduleCode,title,top_skills
23,core,CS1231S,Discrete Structures,"countable, set, infinite, proof, proof infinit..."
24,core,CS2030S,Programming Methodology II,"programming, object oriented, paradigms, objec..."
25,core,CS2040S,Data Structures and Algorithms,"algorithms, data structures, structures, heaps..."
26,core,CS2100,Computer Organisation,"computing devices, data representation, repres..."
27,core,CS2101,Effective Communication for Computing Professi...,"spoken written, engineering equip, reflection ..."
28,core,CS2103T,Software Engineering,"software, modelling design, modelling, object ..."
29,core,CS2106,Introduction to Operating Systems,"memory, memory management, file, file systems,..."
30,core,CS2109S,Introduction to AI and Machine Learning,"search, games, related, related types, local s..."
31,core,CS3230,Design and Analysis of Algorithms,"algorithm, lower, completeness, bound, algorit..."
32,core,MA1521,Calculus for Computing,"calculus, differential equations, differential..."


CS1231S. This course introduces mathematical tools required in the study of computer science. Topics include: (1) Logic and proof techniques: propositions, conditionals, quantifications. (2) Relations and Functions: Equivalence relations and partitions. Partially ordered sets. Well-Ordering Principle. Function equality. Boolean/identity/inverse functions. Bijection. (3) Mathematical formulation of data models (linear model, trees, graphs). (4) Counting and Combinatoric: Pigeonhole Principle. Inclusion-Exclusion Principle. Number of relations on a set, number of injections from one finite set to another, Diagonalization proof: An infinite countable set has an uncountable power set; Algorithmic proof: An infinite set has a countably infinite subset. Subsets of countable sets are countable. Skills: mathematical tools required computer science logic proof techniques propositions conditionals quantifications relations functions equivalence relations partitions partially ordered sets orderin